# 02_procesa_silver

Procesa los registros cargados en `bronze_reportes` para el `id_proceso` actual.

Responsabilidad:
- Leer registros bronze del proceso actual.
- Limpiar valores vacíos o `-`.
- Validar email y fecha de envío.
- Escribir registros válidos en `silver_reportes`.
- Escribir registros inválidos en `errores`.
- Actualizar conteos en `control_archivos`.

Este notebook no usa rutas físicas. Trabaja con tablas Unity Catalog.

In [0]:
# COMMAND ----------

from pyspark.sql import functions as F

In [0]:
# COMMAND ----------
dbutils.widgets.removeAll()

dbutils.widgets.text("catalog_name", "smart_data_dev", "Catalog name")
dbutils.widgets.text("schema_name", "storage", "Schema name")
dbutils.widgets.text("id_proceso", "02ae3d8f-05ac-4af0-86e8-05762dc2db18", "Process id")

catalog_name = dbutils.widgets.get("catalog_name").strip()
schema_name = dbutils.widgets.get("schema_name").strip()
id_proceso = dbutils.widgets.get("id_proceso").strip()

if not all([catalog_name, schema_name, id_proceso]):
    raise ValueError("catalog_name, schema_name e id_proceso son obligatorios.")

tabla_control_sql = f"`{catalog_name}`.`{schema_name}`.`control_archivos`"
tabla_bronze_sql = f"`{catalog_name}`.`{schema_name}`.`bronze_reportes`"
tabla_silver_sql = f"`{catalog_name}`.`{schema_name}`.`silver_reportes`"
tabla_errores_sql = f"`{catalog_name}`.`{schema_name}`.`errores`"

tabla_bronze_name = f"{catalog_name}.{schema_name}.bronze_reportes"
tabla_silver_name = f"{catalog_name}.{schema_name}.silver_reportes"
tabla_errores_name = f"{catalog_name}.{schema_name}.errores"

id_proceso_sql = id_proceso.replace("'", "''")

print(f"catalog_name: {catalog_name}")
print(f"schema_name: {schema_name}")
print(f"id_proceso: {id_proceso}")
print(f"tabla_errores_name: {tabla_errores_name}")

In [0]:
# COMMAND ----------

def limpia_columna_silver(columna):
    return (
        F.when(F.trim(columna).isin("", "-"), F.lit(None))
        .otherwise(F.trim(columna))
    )


fecha_ejecucion = spark.sql("SELECT current_timestamp() AS fecha_ejecucion").first()["fecha_ejecucion"]

columnas_datos = [
    "email",
    "jyv",
    "badmail",
    "baja",
    "fecha_envio",
    "fecha_open",
    "opens",
    "opens_virales",
    "fecha_click",
    "clicks",
    "clicks_virales",
    "links",
    "ips",
    "navegadores",
    "plataformas",
]

In [0]:
# COMMAND ----------

spark.sql(f"DELETE FROM {tabla_silver_sql} WHERE id_proceso = '{id_proceso_sql}'")
spark.sql(f"DELETE FROM {tabla_errores_sql} WHERE id_proceso = '{id_proceso_sql}'")

df_bronze = (
    spark.table(tabla_bronze_name)
    .filter(F.col("id_proceso") == id_proceso)
    .select(
        *[limpia_columna_silver(F.col(columna)).alias(columna) for columna in columnas_datos],
        F.col("_corrupt_record"),
        F.col("_rescued_data"),
        F.col("archivo_origen"),
        F.col("nombre_archivo"),
        F.col("id_proceso"),
    )
)

registros_bronze = df_bronze.limit(1).count()

if registros_bronze == 0:
    mensaje = f"No existen registros bronze para id_proceso={id_proceso}. Silver termina sin cambios."
    print(mensaje)
    dbutils.notebook.exit(mensaje)

print("Registros bronze encontrados para procesar.")

In [0]:
# COMMAND ----------

patron_email = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"

df_validado = (
    df_bronze
    .withColumn("fecha_envio_ts", F.expr("try_to_timestamp(fecha_envio, 'dd/MM/yyyy HH:mm')"))
    .withColumn(
        "fecha_open_ts",
        F.when(F.col("fecha_open").isNull(), F.lit(None).cast("timestamp"))
        .when(
            F.expr("try_to_timestamp(fecha_open, 'dd/MM/yyyy HH:mm')").isNull(),
            F.to_timestamp(F.lit("01/01/2000 00:00"), "dd/MM/yyyy HH:mm")
        )
        .otherwise(F.expr("try_to_timestamp(fecha_open, 'dd/MM/yyyy HH:mm')")),
    )
    .withColumn(
        "fecha_click_ts",
        F.when(F.col("fecha_click").isNull(), F.lit(None).cast("timestamp"))
        .when(
            F.expr("try_to_timestamp(fecha_click, 'dd/MM/yyyy HH:mm')").isNull(),
            F.to_timestamp(F.lit("01/01/2000 00:00"), "dd/MM/yyyy HH:mm")
        )
        .otherwise(F.expr("try_to_timestamp(fecha_click, 'dd/MM/yyyy HH:mm')")),
    )
    .withColumn(
        "razon_error",
        F.when(F.col("_corrupt_record").isNotNull() | F.col("_rescued_data").isNotNull(), F.lit("layout invalido"))
        .when(F.col("email").isNull(), F.lit("sin correo"))
        .when(~F.col("email").rlike(patron_email), F.lit("email incorrecto"))
        .when(F.col("fecha_envio").isNull(), F.lit("fecha_envio vacio"))
        .when(F.col("fecha_envio_ts").isNull(), F.lit("fecha_envio formato invalido"))
        .when(F.col("fecha_envio_ts") > F.current_timestamp(), F.lit("fecha_envio futura"))
        .otherwise(F.lit(None).cast("string")),
    )
)

In [0]:
# COMMAND ----------

df_silver = (
    df_validado
    .filter(F.col("razon_error").isNull())
    .select(
        F.col("email"),
        F.col("jyv"),
        F.col("badmail"),
        F.col("baja"),
        F.col("fecha_envio_ts").alias("fecha_envio"),
        F.col("fecha_open_ts").alias("fecha_open"),
        F.col("opens"),
        F.col("opens_virales"),
        F.col("fecha_click_ts").alias("fecha_click"),
        F.col("clicks"),
        F.col("clicks_virales"),
        F.col("links"),
        F.col("ips"),
        F.col("navegadores"),
        F.col("plataformas"),
        F.col("archivo_origen"),
        F.col("nombre_archivo"),
        F.col("id_proceso"),
        F.lit(fecha_ejecucion).cast("timestamp").alias("fecha_ejecucion"),
        F.date_format(F.col("fecha_envio_ts"), "yyyy-MM").alias("fecha_envio_mes"),
    )
)

df_errores = (
    df_validado
    .filter(F.col("razon_error").isNotNull())
    .select(
        F.col("email"),
        F.col("jyv"),
        F.col("badmail"),
        F.col("baja"),
        F.col("fecha_envio"),
        F.col("fecha_open"),
        F.col("opens"),
        F.col("opens_virales"),
        F.col("fecha_click"),
        F.col("clicks"),
        F.col("clicks_virales"),
        F.col("links"),
        F.col("ips"),
        F.col("navegadores"),
        F.col("plataformas"),
        F.col("archivo_origen"),
        F.col("nombre_archivo"),
        F.col("id_proceso"),
        F.col("_corrupt_record").alias("registro_original"),
        F.col("_rescued_data").alias("rescued_data_column"),
        F.col("razon_error"),
        F.lit(fecha_ejecucion).cast("timestamp").alias("fecha_ejecucion"),
    )
)

(
    df_silver
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(tabla_silver_name)
)

(
    df_errores
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(tabla_errores_name)
)

print("Escritura silver y errores completada.")

In [0]:
# COMMAND ----------

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW tmp_control_silver AS
WITH archivos_proceso AS (
    SELECT DISTINCT nombre_archivo
    FROM {tabla_control_sql}
    WHERE id_proceso = '{id_proceso_sql}'
),
conteo_silver AS (
    SELECT nombre_archivo, COUNT(*) AS registros_silver
    FROM {tabla_silver_sql}
    WHERE id_proceso = '{id_proceso_sql}'
    GROUP BY nombre_archivo
),
conteo_errores AS (
    SELECT nombre_archivo, COUNT(*) AS registros_error
    FROM {tabla_errores_sql}
    WHERE id_proceso = '{id_proceso_sql}'
    GROUP BY nombre_archivo
)
SELECT
    archivos.nombre_archivo,
    COALESCE(silver.registros_silver, 0) AS registros_silver,
    COALESCE(errores.registros_error, 0) AS registros_error
FROM archivos_proceso AS archivos
LEFT JOIN conteo_silver AS silver
    ON archivos.nombre_archivo = silver.nombre_archivo
LEFT JOIN conteo_errores AS errores
    ON archivos.nombre_archivo = errores.nombre_archivo
""")

spark.sql(f"""
MERGE INTO {tabla_control_sql} AS tgt
USING tmp_control_silver AS src
ON tgt.id_proceso = '{id_proceso_sql}'
AND tgt.nombre_archivo = src.nombre_archivo
WHEN MATCHED THEN UPDATE SET
    tgt.registros_silver = src.registros_silver,
    tgt.registros_error = src.registros_error,
    tgt.detalle = NULL
""")

print(f"Proceso silver terminado. id_proceso={id_proceso}")